<a href="https://colab.research.google.com/github/samer-glitch/TADP-Enhancing-Trustworthiness-in-Federated-Learning-Through-Provenance-Driven-Governance/blob/main/11_TADP_HPS28_Contribution_Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
print(os.listdir('/content'))

['.config', 'Fashion_MNIST_HPS28_V5_Results.zip', 'TADP_HPS28_Adult_Phased_Results_v5.zip', 'sample_data']


In [4]:
#!/usr/bin/env python3
"""TADP HPS-28: audit the ACTUAL archived decision and round records.

This is a post-hoc diagnostic, NOT retraining. It does not authenticate evidence,
validate HPS against a real-world criterion, or change the original policy.

Usage:
  python TADP_HPS28_Contribution_Audit.py \
     --adult TADP_HPS28_Adult_Phased_Results_v5.zip \
     --fashion Fashion_MNIST_HPS28_V5_Results.zip \
     --out TADP_HPS28_Contribution_Audit

Outputs: policy_stage_audit.csv, counterfactual_policy_ablation.csv,
         aggregation_weight_audit.csv, paired_macro_f1.csv,\n         degradation_decision_transitions.csv, AUDIT_README.md.
"""
from __future__ import annotations
import argparse
import csv
import json
import re
import statistics
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

FACTORS = {
    'D1': ('source_reputation','data_controller','data_objective','data_collection_lineage'),
    'D2': ('completeness','duplication_rate','value_validity_error_rate','type_consistency',
           'label_integrity','feature_distribution_consistency','feature_category_coverage',
           'structural_constraint_integrity'),
    'D3': ('data_dictionary','version_logs','collection_protocol','definition_updates'),
    'D4': ('data_freshness','scheduled_refresh','retention_clarity'),
    'D5': ('regulation_coverage','consent_ethics','geo_restrictions','sensitivity_classification','audits'),
    'D6': ('license_terms','ethical_reviews','redistribution','user_agreements'),
}
WEIGHTS = {'D1':.25,'D2':.15,'D3':.10,'D4':.10,'D5':.30,'D6':.10}
DOC_FACTORS = set().union(*(set(f) for d,f in FACTORS.items() if d!='D2'))
D2_FACTORS = set(FACTORS['D2'])
ALL_FACTORS = DOC_FACTORS | D2_FACTORS
GOVERNED = ('HPG', 'PWA', 'WUPW')

def write_rows(path, rows):
    if not rows: raise ValueError('No results for '+str(path))
    keys=list(rows[0]);
    with open(path,'w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,keys);writer.writeheader();writer.writerows(rows)

def load_zip(path):
    z=zipfile.ZipFile(path)
    decision_paths=sorted(n for n in z.namelist() if '/decisions_' in n and n.endswith('_none.json'))
    round_paths=sorted(n for n in z.namelist() if '/rounds_' in n and n.endswith('_none.json'))
    result_paths=sorted(n for n in z.namelist() if '/results_' in n and n.endswith('_none.csv'))
    if len(decision_paths)!=20 or len(round_paths)!=20 or len(result_paths)!=20:
        raise ValueError(f'Expected exactly 20 unperturbed decision/round/result files; got '
                         f'{len(decision_paths)}/{len(round_paths)}/{len(result_paths)} in {path}')
    def key(name):
        alpha=re.search(r'alpha(0\.[0-9]+)',name)
        seed=re.search(r'_seed(4[2-6])_',name)
        if not alpha or not seed: raise ValueError('Unexpected path '+name)
        return (alpha.group(1), int(seed.group(1)), 'degraded' if '_degraded_none.' in name else 'clean')
    d={key(p):json.loads(z.read(p)) for p in decision_paths}
    r={key(p):json.loads(z.read(p)) for p in round_paths}
    v={key(p):list(csv.DictReader(z.read(p).decode('utf-8').splitlines())) for p in result_paths}
    if set(d)!=set(r) or set(d)!=set(v):raise ValueError('Decision/round/result condition mismatch')
    if any(len(x['decisions'])!=50 for x in d.values()):raise ValueError('Client count mismatch')
    return d,r,v

def policy_flags(d, policy):
    scores=d['factor_scores']; dims=d['dimensions']
    if set(scores)!=ALL_FACTORS or set(dims)!=set(FACTORS): raise ValueError('Factor mismatch')
    has_d2_zero=any(scores[f]==0 for f in D2_FACTORS)
    has_doc_zero=any(scores[f]==0 for f in DOC_FACTORS)
    zero=has_d2_zero or has_doc_zero
    floor=any(v<policy['min_dimension'] for v in dims.values())
    critical=any(scores[f]<limit for f,limit in policy.get('critical_factor_floors',{}).items())
    hps=sum(WEIGHTS[d]*dims[d] for d in FACTORS)
    review=sum(WEIGHTS[x]*dims[x] for x in ('D1','D2','D5','D6'))/.80
    if abs(hps-d['hps'])>1e-7 or abs(review-d['hps_review'])>1e-7:
        raise ValueError('Archived HPS and recomputed HPS differ')
    numerical=(hps>=policy['direct_accept_from'] or
               (hps>=policy['reject_below'] and
                review>=(policy['reject_below']+policy['direct_accept_from'])/2))
    all_pass=(not zero and not floor and not critical and numerical)
    if all_pass!=d['accepted']:
        raise ValueError(f'Archived decision differs from reconstructed decision: {d["path"]}')
    return {'zero':zero,'d2_zero':has_d2_zero,'documentary_zero':has_doc_zero,
            'floor':floor,'critical':critical,'numerical':numerical,'full':all_pass}

def sum_round(x):return sum(x)

def mean_sd(x):
    return (statistics.mean(x),statistics.stdev(x) if len(x)>1 else 0.0)

def audit(adult, fashion, out):
    out.mkdir(parents=True,exist_ok=True)
    stage_rows=[]; ablation_rows=[]; weight_rows=[]; predictive_rows=[]; transition_rows=[]
    combined=Counter()
    for dataset,zip_path in (('Adult',adult),('Fashion-MNIST',fashion)):
        decisions,rounds,results=load_zip(zip_path)
        for alpha in ('0.5','0.1'):
            for seed in (42,43,44,45,46):
                orig=decisions[(alpha,seed,'clean')]['decisions']
                altered=decisions[(alpha,seed,'degraded')]['decisions']
                for group, subset in (
                    ('targeted_for_degradation',[cid for cid in orig if altered[cid]['degraded']]),
                    ('not_targeted',[cid for cid in orig if not altered[cid]['degraded']]),
                ):
                    transition_rows.append({
                        'dataset':dataset,'alpha':alpha,'seed':seed,'group':group,
                        'n_clients':len(subset),
                        'admitted_clean':sum(orig[cid]['accepted'] for cid in subset),
                        'admitted_degraded':sum(altered[cid]['accepted'] for cid in subset),
                        'accepted_to_rejected':sum(orig[cid]['accepted'] and not altered[cid]['accepted'] for cid in subset),
                        'rejected_to_accepted':sum(not orig[cid]['accepted'] and altered[cid]['accepted'] for cid in subset),
                    })
        for (alpha,seed,condition),payload in sorted(decisions.items()):
            policy=payload['policy']
            flags={cid:policy_flags(decision,policy) for cid,decision in payload['decisions'].items()}
            paths=Counter(d['path'] for d in payload['decisions'].values())
            for cid, d in payload['decisions'].items():
                st=flags[cid]
                combined['assessments']+=1
                if d['accepted']:combined['accepted']+=1
                if d['path'] in ('BLOCKED_ZERO_FACTOR','BLOCKED_DIMENSION_FLOOR','BLOCKED_CRITICAL_FACTOR'):
                    combined['mandatory_rejected']+=1
                if not d['accepted'] and d['path'] in ('AUTO_REJECT','REVIEW_REJECT'):
                    combined['numerical_additional_rejected']+=1
                if not d['accepted'] and d['hps']>=policy['direct_accept_from']:
                    combined['rejected_high_score']+=1
                if st['d2_zero']: combined['cases_d2_zero']+=1
                if st['documentary_zero']:combined['cases_doc_zero']+=1
                if st['d2_zero'] and d['factor_scores']['feature_distribution_consistency']==0:
                    combined['cases_distribution_zero']+=1
            stage_rows.append({
                'dataset':dataset,'alpha':alpha,'seed':seed,'condition':condition,
                'clients':len(flags),'admitted':sum(x['full'] for x in flags.values()),
                'zero_stage_rejected':paths['BLOCKED_ZERO_FACTOR'],
                'dimension_stage_rejected':paths['BLOCKED_DIMENSION_FLOOR'],
                'other_mandatory_rejected':paths['BLOCKED_DISQUALIFIER']+paths['BLOCKED_CRITICAL_FACTOR'],
                'hps_below_rejection_threshold':paths['AUTO_REJECT'],
                'review_stage_rejected':paths['REVIEW_REJECT'],
                'review_stage_accepted':paths['REVIEW_ACCEPT'],
                'direct_accept':paths['AUTO_ACCEPT'],
                'd2_zero_cases_including_overlaps':sum(x['d2_zero'] for x in flags.values()),
                'doc_zero_cases_including_overlaps':sum(x['documentary_zero'] for x in flags.values()),
                'feature_distribution_consistency_zero':sum(d['factor_scores']['feature_distribution_consistency']==0
                      for d in payload['decisions'].values()),
                'rejected_despite_HPS_at_least_direct_accept':sum(not d['accepted'] and
                          d['hps']>=policy['direct_accept_from'] for d in payload['decisions'].values()),
                'degraded_clients':sum(d.get('degraded',False) for d in payload['decisions'].values()),
                'degraded_rejected':sum(d.get('degraded',False) and not d['accepted'] for d in payload['decisions'].values()),
                'nondegraded_rejected':sum(not d.get('degraded',False) and not d['accepted'] for d in payload['decisions'].values()),
            })
            definitions={
                'full_original':lambda x:x['full'],
                'mandatory_rules_only':lambda x:not (x['zero'] or x['floor'] or x['critical']),
                'zero_veto_only':lambda x:not x['zero'],
                'no_zero_veto':lambda x:not (x['floor'] or x['critical']) and x['numerical'],
                'no_dimension_floor':lambda x:not (x['zero'] or x['critical']) and x['numerical'],
                'numerical_HPS_only':lambda x:x['numerical'],
            }
            for name,fn in definitions.items():
                admitted={cid for cid,x in flags.items() if fn(x)}
                originally={cid for cid,x in flags.items() if x['full']}
                ablation_rows.append({
                    'dataset':dataset,'alpha':alpha,'seed':seed,'condition':condition,
                    'diagnostic_policy':name,'admitted':len(admitted),
                    'gained_vs_original':len(admitted-originally),
                    'lost_vs_original':len(originally-admitted),
                    'decision_changes_vs_original':len(admitted.symmetric_difference(originally)),
                    'note':'Counterfactual diagnostic only; not a recommended deployment policy'
                })
            by_method_round=defaultdict(dict)
            for row in rounds[(alpha,seed,condition)]:
                if row['method'] in GOVERNED:
                    by_method_round[row['round']][row['method']]=row
            for round_number,rows in sorted(by_method_round.items()):
                if set(rows)!=set(GOVERNED):raise ValueError('Missing HPG/PWA/WUPW rows')
                base=rows['HPG']
                ids=set(base['selected_ids']); expected=sum(float(w) for w in base['weights'].values())
                if abs(expected-1)>1e-8:raise ValueError('HPG weights do not sum to one')
                for name in ('PWA','WUPW'):
                    alternative=rows[name]
                    if ids!=set(alternative['selected_ids']):
                        raise ValueError('Cohort/selected IDs do not match in shared round')
                    if abs(sum(alternative['weights'].values())-1)>1e-8:
                        raise ValueError('Aggregated weights do not sum to one')
                    L1=sum(abs(base['weights'][cid]-alternative['weights'][cid]) for cid in ids)
                    hps=[payload['decisions'][cid]['hps'] for cid in ids]
                    weight_rows.append({
                        'dataset':dataset,'alpha':alpha,'seed':seed,'condition':condition,
                        'round':round_number,'comparison':f'{name}_vs_HPG',
                        'n_same_selected_clients':len(ids),
                        'total_variation_distance':round(L1/2,10),
                        'L1_weight_distance':round(L1,10),
                        'minimum_selected_HPS':round(min(hps),8),
                        'maximum_selected_HPS':round(max(hps),8),
                        'selected_HPS_spread':round(max(hps)-min(hps),8),
                        'same_selected_client_ids':True,
                        'same_admission_cohort':True,
                    })
            metrics={row['method']:row for row in results[(alpha,seed,condition)]}
            if set(GOVERNED)-set(metrics):raise ValueError('Missing governed method metrics')
            for name in ('PWA','WUPW'):
                predictive_rows.append({
                    'dataset':dataset,'alpha':alpha,'seed':seed,'condition':condition,
                    'comparison':f'{name}_minus_HPG',
                    'HPG_macro_f1':round(float(metrics['HPG']['macro_f1']),8),
                    'alternative_macro_f1':round(float(metrics[name]['macro_f1']),8),
                    'macro_f1_difference':round(float(metrics[name]['macro_f1'])-float(metrics['HPG']['macro_f1']),8),
                    'HPG_cohort_size':int(float(metrics['HPG']['cohort_size'])),
                    'alternative_cohort_size':int(float(metrics[name]['cohort_size'])),
                })
    write_rows(out/'policy_stage_audit.csv',stage_rows)
    write_rows(out/'counterfactual_policy_ablation.csv',ablation_rows)
    write_rows(out/'aggregation_weight_audit.csv',weight_rows)
    write_rows(out/'paired_macro_f1.csv',predictive_rows)
    write_rows(out/'degradation_decision_transitions.csv',transition_rows)
    by_compare=defaultdict(list)
    for r in weight_rows:by_compare[(r['dataset'],r['comparison'])].append(r['total_variation_distance'])
    all_mand=combined['mandatory_rejected']; total_rejected=combined['assessments']-combined['accepted']
    report=[
        '# TADP HPS-28 — contribution and policy audit',
        '',
        'These results are recomputed from archived decision, round and outcome records.',
        'No federated training is rerun. No independent provenance evidence is validated.',
        '',
        f'- Client assessments (repeated across experimental conditions): **{combined["assessments"]}**',
        f'- Admitted: **{combined["accepted"]}**; rejected: **{total_rejected}**',
        f'- Rejected at mandatory stages (zero factors / floors): **{all_mand} / {total_rejected}** '
        f'({100*all_mand/total_rejected:.2f}%)',
        f'- Additional numerical HPS / intermediate-review rejections: **{combined["numerical_additional_rejected"]}** '
        f'({100*combined["numerical_additional_rejected"]/combined["assessments"]:.3f}% of assessed cases)',
        f'- Rejected even though numerical HPS was >= 3.5: **{combined["rejected_high_score"]}** '
        '(mandatory safeguards take precedence)',
        f'- Cases with at least one D2 zero: **{combined["cases_d2_zero"]}**; '
        f'with at least one documentary zero: **{combined["cases_doc_zero"]}** '
        '(can overlap)',
        f'- Cases with zero for feature-distribution consistency: **{combined["cases_distribution_zero"]}**',
        '',
        '## Original-weight changes (same round participants in all comparisons)',
        '',
        '| Dataset | Pair | Mean total variation | Maximum total variation | Rounds |',
        '|---|---|---:|---:|---:|',
    ]
    for k,v in sorted(by_compare.items()):
        report.append(f'| {k[0]} | {k[1]} | {statistics.mean(v):.4f} | {max(v):.4f} | {len(v)} |')
    report += ['',
      'Total variation = 0.5 × L1 distance of normalized aggregation-weight vectors.',
      'It measures how much weights change, not model-performance improvement.',
      '', '## Interpretation and cautions', '',
      '1. The 28-factor numerical score governs score-adjusted aggregation even if only a few additional rejections arise at the numeric threshold/review stages.',
      '2. Because mandatory vetoes dominate admissions, numerical HPS admission value requires independent demonstration.',
      '3. The archived HPG, PWA and WUPW rounds used exactly the same selected client IDs; thus their comparative FL evaluation already controls cohort membership, although training trajectories diverge after applying different weights.',
      '4. Counterfactual policies are **diagnostic**. Relaxed safeguards may admit unsuitable clients and must not be presented as safer alternatives.',
      '5. Dataset-level corruption is injected synthetically; degraded status is not a real-world eligibility ground truth.',
      '6. Repeated conditions and seeds do not represent independent institutions or datasets.',
      '', '## Generated files', '',
      '- `policy_stage_audit.csv`: exact policy-stage paths and high-HPS rejection counts by seed and condition',
      '- `counterfactual_policy_ablation.csv`: rule ablations recomputed without retraining',
      '- `aggregation_weight_audit.csv`: matched per-round aggregation weights',
      '- `paired_macro_f1.csv`: within-seed PWA/WUPW vs HPG macro F1 differences',
      '', '### Provenance', '',
      f'- Adult archive: `{adult.name}`',
      f'- Fashion archive: `{fashion.name}`',
      '- Read-only ZIP input; code checks saved HPS computations and admission decisions against the original policy.',
    ]
    (out/'AUDIT_README.md').write_text('\n'.join(report)+'\n',encoding='utf-8')
    print('\n'.join(report[:31]))
    return out


# ============================================================================
# SIMPLE COLAB EXECUTION: upload this file + the two original results ZIPs.
# In a Colab cell: %run 01_Run_HPS_Contribution_Audit.py
# ============================================================================

def _run_one_click():
    from pathlib import Path
    import zipfile
    import os

    root = Path('/content') if Path('/content').is_dir() else Path.cwd()
    def find_zip(stem):
        candidates = sorted(root.glob(stem + '*.zip'))
        preferred = root / (stem + '.zip')
        if preferred.exists():
            return preferred
        if len(candidates) != 1:
            raise FileNotFoundError('Upload ' + stem + '.zip in the Colab Files panel; found: ' + str(candidates))
        return candidates[0]

    adult = find_zip('TADP_HPS28_Adult_Phased_Results_v5')
    fashion = find_zip('Fashion_MNIST_HPS28_V5_Results')
    result_dir = root/'STEP1_HPS_Contribution_Results'
    print('EXPERIMENT 1: HPS CONTRIBUTION AUDIT (no model training)')
    print('Using:', adult.name, 'and', fashion.name, flush=True)
    audit(adult, fashion, result_dir)
    bundle = root/'STEP1_HPS_Contribution_Results.zip'
    with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for entry in sorted(result_dir.rglob('*')):
            if entry.is_file():
                z.write(entry, entry.relative_to(result_dir.parent))
    print('SUCCESS. Download:', str(bundle), flush=True)
    print('In Colab, click the Files (folder) icon and download the ZIP.');
    return bundle

if __name__ == '__main__':
    _run_one_click()


EXPERIMENT 1: HPS CONTRIBUTION AUDIT (no model training)
Using: TADP_HPS28_Adult_Phased_Results_v5.zip and Fashion_MNIST_HPS28_V5_Results.zip
# TADP HPS-28 — contribution and policy audit

These results are recomputed from archived decision, round and outcome records.
No federated training is rerun. No independent provenance evidence is validated.

- Client assessments (repeated across experimental conditions): **2000**
- Admitted: **1164**; rejected: **836**
- Rejected at mandatory stages (zero factors / floors): **833 / 836** (99.64%)
- Additional numerical HPS / intermediate-review rejections: **3** (0.150% of assessed cases)
- Rejected even though numerical HPS was >= 3.5: **424** (mandatory safeguards take precedence)
- Cases with at least one D2 zero: **607**; with at least one documentary zero: **100** (can overlap)
- Cases with zero for feature-distribution consistency: **449**

## Original-weight changes (same round participants in all comparisons)

| Dataset | Pair | Mean tot

In [6]:
!wget -q https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data -O /content/adult.data

!wc -l /content/adult.data

32562 /content/adult.data


In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""STEP 2: isolated Adult data-defect response, without FL retraining.

Uses the ORIGINAL 28-factor TADP policy and Adult D2 functions copied from the
November 10 benchmark source. Verifies the supplied Adult dataset/split against
existing archived decision fingerprints before any analysis. No TEST labels are
used for scoring or calibration. Runs missingness, duplication, invalid numeric
values, and within-class-set label flips SEPARATELY for original targeted clients.

Outputs are controlled synthetic-scenario sensitivity; they do NOT authenticate
legal provenance or prove admission accuracy. Full HPS documentary factors are
held fixed, and unmodified zero-factor rejections are reported distinctly.
"""
from __future__ import annotations
import argparse
import csv
import hashlib
import json
import math
from collections import defaultdict, Counter
from dataclasses import dataclass, field, asdict
from math import isfinite
from numbers import Real
from pathlib import Path
from types import SimpleNamespace
from typing import Mapping, Any
from zipfile import ZipFile
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

POLICY_VERSION = "TADP-HPS-28/1.0"
FACTORS = {
 "D1": ("source_reputation", "data_controller", "data_objective", "data_collection_lineage"),
 "D2": ("completeness", "duplication_rate", "value_validity_error_rate", "type_consistency",
        "label_integrity", "feature_distribution_consistency", "feature_category_coverage",
        "structural_constraint_integrity"),
 "D3": ("data_dictionary", "version_logs", "collection_protocol", "definition_updates"),
 "D4": ("data_freshness", "scheduled_refresh", "retention_clarity"),
 "D5": ("regulation_coverage", "consent_ethics", "geo_restrictions", "sensitivity_classification", "audits"),
 "D6": ("license_terms", "ethical_reviews", "redistribution", "user_agreements"),
}
WEIGHTS = {"D1": .25, "D2": .15, "D3": .10, "D4": .10, "D5": .30, "D6": .10}
CRITICAL = ("D1", "D2", "D5", "D6")
ALL_FACTORS = frozenset(f for fs in FACTORS.values() for f in fs)
D2_FACTORS = FACTORS["D2"]
_VIOLATION_CUTS = {
 "completeness": (.50, .20, .10, .05, .01),
 "duplication_rate": (.20, .10, .05, .02, .01),
 "value_validity_error_rate": (.15, .10, .05, .02, .01),
 "type_consistency": (.20, .10, .05, .02, .01),
 "label_integrity": (.10, .05, .02, .01, .001),
 "feature_distribution_consistency": (.20, .10, .05, .025, .01),
 "structural_constraint_integrity": (.10, .05, .02, .01, .001),
}
ADULT_COLUMNS = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
 "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]
ADULT_NUMERIC = ("age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week")
NUMERIC_RANGES = {"age": (1, 120), "fnlwgt": (1, 1e10), "education-num": (1, 16),
 "capital-gain": (0, 1e9), "capital-loss": (0, 1e9), "hours-per-week": (0, 168)}


# SOURCE-ALIGNED FUNCTION: Policy
@dataclass(frozen=True)
class Policy:
    min_dimension: float = 2.5
    reject_below: float = 3.0
    direct_accept_from: float = 3.5
    # Explicit individual critical-factor safeguards, if legally/operationally justified.
    critical_factor_floors: Mapping[str, int] = field(default_factory=dict)
    policy_version: str = POLICY_VERSION

    @property
    def review_midpoint(self):
        return (self.reject_below + self.direct_accept_from) / 2.0

    def validate(self):
        bounds = (self.min_dimension, self.reject_below, self.direct_accept_from)
        if not all(isfinite(v) and 0 <= v <= 5 for v in bounds):
            raise ValueError("Invalid policy thresholds")
        if not (self.min_dimension <= self.reject_below < self.direct_accept_from):
            raise ValueError("Require minimum dimension <= reject threshold < accept threshold")
        for f, threshold in self.critical_factor_floors.items():
            if f not in ALL_FACTORS or type(threshold) is not int or not 0 <= threshold <= 5:
                raise ValueError(f"Invalid critical-factor floor: {f}")

# SOURCE-ALIGNED FUNCTION: Decision
@dataclass(frozen=True)
class Decision:
    accepted: bool
    path: str
    hps: float | None
    hps_review: float | None
    dimensions: dict[str, float]
    reasons: tuple[str, ...]
    factor_scores: dict[str, int]
    policy_version: str

    def record(self) -> dict:
        return asdict(self)

# SOURCE-ALIGNED FUNCTION: evaluate
def evaluate(scores: Mapping[str, int], policy: Policy = Policy(),
             disqualifiers: tuple[str, ...] = (),
             ablate_distribution: bool = False) -> Decision:
    policy.validate()
    missing = sorted(ALL_FACTORS - scores.keys())
    unknown = sorted(scores.keys() - ALL_FACTORS)
    if missing or unknown:
        return Decision(False, "BLOCKED_INCOMPLETE", None, None, {},
                        (f"Missing={missing}, unexpected={unknown}",), dict(scores), policy.policy_version)
    for factor, value in scores.items():
        if type(value) is not int or not 0 <= value <= 5:
            return Decision(False, "BLOCKED_INVALID", None, None, {},
                            (f"Invalid rubric score: {factor}={value!r}",), dict(scores), policy.policy_version)
    factor_groups = dict(FACTORS)
    if ablate_distribution:
        factor_groups["D2"] = tuple(f for f in FACTORS["D2"]
                                     if f not in ("feature_distribution_consistency","feature_category_coverage"))
    dimensions = {d: sum(scores[f] for f in fs)/len(fs) for d, fs in factor_groups.items()}
    hps = sum(WEIGHTS[d]*dimensions[d] for d in FACTORS)
    den = sum(WEIGHTS[d] for d in CRITICAL)
    hps_review = sum(WEIGHTS[d]*dimensions[d] for d in CRITICAL)/den
    def out(accepted, path, *reason):
        version = policy.policy_version + ("/NoDistributionAblation" if ablate_distribution else "")
        return Decision(accepted, path, round(hps, 10), round(hps_review, 10),
                        dimensions, tuple(reason), dict(scores), version)
    if disqualifiers:
        return out(False, "BLOCKED_DISQUALIFIER", *disqualifiers)
    active = set(f for fs in factor_groups.values() for f in fs)
    zeros = sorted(f for f, v in scores.items() if f in active and v == 0)
    if zeros:
        return out(False, "BLOCKED_ZERO_FACTOR", f"Required factor(s) scored zero: {zeros}")
    low = [f"{d}={v:.3f} < {policy.min_dimension}" for d, v in dimensions.items()
           if v < policy.min_dimension]
    if low:
        return out(False, "BLOCKED_DIMENSION_FLOOR", *low)
    for f, floor in policy.critical_factor_floors.items():
        if scores[f] < floor:
            return out(False, "BLOCKED_CRITICAL_FACTOR", f"{f}={scores[f]} < {floor}")
    if hps < policy.reject_below:
        return out(False, "AUTO_REJECT", f"HPS={hps:.4f} < {policy.reject_below}")
    if hps >= policy.direct_accept_from:
        return out(True, "AUTO_ACCEPT", f"HPS={hps:.4f} >= {policy.direct_accept_from}")
    if hps_review >= policy.review_midpoint:
        return out(True, "REVIEW_ACCEPT", f"HPS_review={hps_review:.4f} >= {policy.review_midpoint}")
    return out(False, "REVIEW_REJECT", f"HPS_review={hps_review:.4f} < {policy.review_midpoint}")

# SOURCE-ALIGNED FUNCTION: _ks_max
def _ks_max(a, b, seed=1, cap=10000):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a).reshape(-1), np.asarray(b).reshape(-1)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if not len(a) or not len(b): return 1.0
    if len(a) > cap: a = rng.choice(a, cap, replace=False)
    if len(b) > cap: b = rng.choice(b, cap, replace=False)
    return float(ks_2samp(a, b).statistic)

# SOURCE-ALIGNED FUNCTION: adult_train_metrics
def adult_train_metrics(client_x: pd.DataFrame, labels, reference_x: pd.DataFrame):
    if client_x.empty: raise ValueError("No client training records")
    if not set(ADULT_NUMERIC) <= set(client_x.columns): raise ValueError("Adult schema mismatch")
    if set(client_x.columns) != set(reference_x.columns): raise ValueError("Reference schema mismatch")
    x = client_x.replace(r"^\s*\?\s*$", np.nan, regex=True).replace(r"^\s*$", np.nan, regex=True)
    ref = reference_x.replace(r"^\s*\?\s*$", np.nan, regex=True).replace(r"^\s*$", np.nan, regex=True)
    numeric, categorical = list(ADULT_NUMERIC), [c for c in x.columns if c not in ADULT_NUMERIC]
    n, c = x.shape
    completeness = float(x.isna().to_numpy().mean())
    duplication = float(x.duplicated().mean())
    numeric_arr = x[numeric].apply(pd.to_numeric, errors="coerce")
    num_observed = x[numeric].notna().to_numpy()
    numeric_type_invalid = np.logical_and(num_observed, numeric_arr.isna().to_numpy()).sum()
    # Expected categorical fields are strings (allow object/string and reject unexpected list-like data).
    cat_type_bad = sum(sum(not isinstance(v, (str, np.str_)) for v in x[col].dropna()) for col in categorical)
    observed_count = int(x.notna().to_numpy().sum())
    type_inconsistency = (numeric_type_invalid + cat_type_bad) / max(observed_count, 1)
    num_invalid = np.zeros((n, len(numeric)), dtype=bool)
    bad_row = np.zeros(n, dtype=bool)
    for idx, field in enumerate(numeric):
        vals = numeric_arr[field].to_numpy(dtype=float)
        lo, hi = NUMERIC_RANGES[field]
        bad = np.isfinite(vals) & ((vals < lo) | (vals > hi))
        num_invalid[:, idx] = bad
        bad_row |= bad
    # Numeric type errors also count as invalid values, independently of completeness.
    validity = float((num_invalid.sum() + numeric_type_invalid) / max(observed_count, 1))
    structural = float(bad_row.mean())
    ys = np.asarray(labels)
    if len(ys) != n: raise ValueError("Label count mismatch")
    label_bad = float((~np.isin(ys, [0, 1])).mean())
    ks_vals = []
    for k, field in enumerate(numeric):
        r = pd.to_numeric(ref[field], errors="coerce").to_numpy(dtype=float)
        p = numeric_arr[field].to_numpy(dtype=float)
        ks_vals.append(_ks_max(p, r, seed=1337+k))
    consistency = max(ks_vals)
    coverage = []
    for field in categorical:
        ref_levels = set(ref[field].dropna().astype(str).str.strip().str.lower())
        present = set(x[field].dropna().astype(str).str.strip().str.lower())
        if ref_levels: coverage.append(len(ref_levels & present)/len(ref_levels))
    return dict(completeness=completeness, duplication_rate=duplication,
                value_validity_error_rate=min(1.,validity), type_consistency=min(1.,float(type_inconsistency)),
                label_integrity=label_bad, feature_distribution_consistency=float(consistency),
                feature_category_coverage=float(np.mean(coverage) if coverage else 1),
                structural_constraint_integrity=structural)

# SOURCE-ALIGNED FUNCTION: d2_from_metrics
def d2_from_metrics(metrics: Mapping[str, float]) -> dict[str, int]:
    if set(metrics) != set(D2_FACTORS):
        raise ValueError(f"Expected all eight D2 factors, missing={set(D2_FACTORS)-set(metrics)}")
    result = {}
    for name, value in metrics.items():
        if isinstance(value, bool) or not isinstance(value, Real) or not isfinite(value) or not 0 <= value <= 1:
            raise ValueError(f"Invalid TRAIN-only metric {name}={value!r}")
        if name == "feature_category_coverage":
            if value < .50: score = 0
            elif value < .65: score = 1
            elif value < .75: score = 2
            elif value < .825: score = 3
            elif value < .90: score = 4
            else: score = 5
        else:
            score = next((i for i, cut in enumerate(_VIOLATION_CUTS[name]) if value > cut), 5)
        result[name] = score
    return result

# SOURCE-ALIGNED FUNCTION: load_dataset
def load_dataset(dataset, adult_csv=None, fashion_root="./data", download=True):
    if dataset == "adult":
        if not adult_csv: raise ValueError("Provide --adult-csv path to Adult CSV (raw with missing values)")
        raw = pd.read_csv(adult_csv, skipinitialspace=True, na_values=["?", " ?", "", "nan"])
        if raw.shape[1] == 15 and str(raw.columns[0]).lower() != "age":
            # Headerless UCI adult.data file: preserve first data row.
            raw = pd.read_csv(adult_csv, header=None, skipinitialspace=True,
                              na_values=["?", " ?", "", "nan"])
        if raw.shape[1] != 15: raise ValueError("Adult CSV must have 15 columns")
        if list(raw.columns) != ADULT_COLUMNS:
            if ("income" in [str(c).lower() for c in raw.columns] or len(raw.columns)==15):
                raw.columns = ADULT_COLUMNS
        raw["income"] = raw["income"].astype(str).str.strip().str.lower().str.rstrip(".")
        raw = raw[raw["income"].isin(["<=50k", ">50k"])].copy()
        if raw.empty: raise ValueError("No recognized Adult binary income labels")
        y = (raw["income"] == ">50k").astype(np.int64).to_numpy()
        return raw.drop(columns="income").reset_index(drop=True), y
    try:
        from torchvision.datasets import FashionMNIST
    except Exception as exc:
        raise RuntimeError("torchvision is needed for Fashion-MNIST") from exc
    tr = FashionMNIST(root=fashion_root, train=True, download=download)
    te = FashionMNIST(root=fashion_root, train=False, download=download)
    return (tr.data.numpy(), tr.targets.numpy(), te.data.numpy(), te.targets.numpy())

# SOURCE-ALIGNED FUNCTION: split_global
def split_global(dataset, loaded, cfg: Config, seed):
    """One immutable TRAIN/TEST division across ALL experimental seeds and methods.

    Adult has no patient identifier: claims of patient-grouped splitting are INVALID.
    Instead, identical raw predictor records are grouped to prevent exact
    duplicate predictors occurring in both TRAIN and TEST. Approximate 80/20,
    stratified as closely as possible under those grouping constraints.

    D2-only reference uses GLOBAL TRAIN predictors; it is never fitted as model
    preprocessing and is never used for gradient updates separately.
    """
    if dataset == "adult":
        from sklearn.model_selection import StratifiedGroupKFold
        x, y = loaded
        groups = pd.util.hash_pandas_object(x.fillna("__RAW_MISSING__").astype(str), index=False).to_numpy()
        splits = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=cfg.global_split_seed)
        target_frac = cfg.test_fraction
        global_rate = float(np.mean(y))
        candidates = []
        for train_index, test_index in splits.split(x, y, groups):
            frac = len(test_index) / len(y)
            rate = float(np.mean(y[test_index]))
            candidates.append((abs(frac-target_frac)+abs(rate-global_rate), train_index, test_index))
        _, train_index, test_index = min(candidates, key=lambda z:z[0])
        if set(groups[train_index]) & set(groups[test_index]):
            raise AssertionError("LEAKAGE: identical raw predictor group occurs in TRAIN and TEST")
        xtrain = x.iloc[train_index].reset_index(drop=True)
        xtest = x.iloc[test_index].reset_index(drop=True)
        ytrain, ytest = np.asarray(y)[train_index], np.asarray(y)[test_index]
        assert len(xtrain)+len(xtest)==len(x)
        assert not (set(groups[train_index]) & set(groups[test_index]))
        return xtrain, ytrain, xtrain, xtest, ytest
    xtrain,ytrain,xtest,ytest = loaded
    return xtrain,np.asarray(ytrain),xtrain,xtest,np.asarray(ytest)

# SOURCE-ALIGNED FUNCTION: dirichlet_partition
def dirichlet_partition(y, n_clients, alpha, seed, min_samples=16, max_repair_fraction=.10,
                        return_diagnostics=False):
    """Seeded label-wise Dirichlet allocation plus EXPLICIT minimum-size repair.

    Rejection sampling with K=50, alpha=0.1 is rarely feasible. Instead sample
    the requested Dirichlet ONCE, then relocate the minimum records required
    from largest donors into undersized clients, preserving class labels.
    Because the repair changes the sampled allocation, report its count/rate;
    never describe the final partition as an unmodified Dirichlet draw.
    """
    rng=np.random.default_rng(seed)
    y=np.asarray(y)
    if alpha<=0 or len(y)<n_clients*min_samples:
        raise ValueError("Impossible partition: alpha<=0 or insufficient TRAIN records")
    bags=[[] for _ in range(n_clients)]
    for label in np.unique(y):
        indices=rng.permutation(np.flatnonzero(y==label))
        proportions=rng.dirichlet(np.full(n_clients, alpha))
        counts=rng.multinomial(len(indices), proportions)
        for i,part in enumerate(np.split(indices, np.cumsum(counts)[:-1])):
            bags[i].extend(part.tolist())
    pre_min=min(map(len,bags))
    moved=0
    for receiver in sorted(range(n_clients),key=lambda i:(len(bags[i]),i)):
        while len(bags[receiver]) < min_samples:
            donors=[j for j in range(n_clients) if len(bags[j])>min_samples]
            if not donors:raise ValueError("Partition quota infeasible; no donor clients")
            donor=max(donors,key=lambda j:(len(bags[j]),-j))
            j=int(rng.integers(len(bags[donor])))
            bags[receiver].append(bags[donor].pop(j));moved+=1
    assigned=np.concatenate([np.asarray(b,dtype=int) for b in bags])
    if len(assigned)!=len(y) or len(set(assigned.tolist()))!=len(y):
        raise AssertionError("Partition has dropped or duplicated TRAIN records")
    repair_fraction=moved/len(y)
    if repair_fraction>max_repair_fraction:
        raise ValueError(f"Partition repair {repair_fraction:.2%} > prespecified cap {max_repair_fraction:.2%}")
    diag={"alpha_requested":float(alpha),"seed":int(seed),"n_clients":n_clients,
          "min_client_size_before_repair":int(pre_min),"min_client_size_after_repair":int(min(map(len,bags))),
          "records_reallocated":int(moved),"repair_fraction":float(repair_fraction),
          "min_samples":int(min_samples),"distribution":"Dirichlet with logged minimum-client-size repair"}
    indices=[np.array(sorted(b),dtype=int) for b in bags]
    return (indices,diag) if return_diagnostics else indices

# SOURCE-ALIGNED FUNCTION: split_fingerprint
def split_fingerprint(xtr,ytr,xt,yt):
    """Bind cached results to the full, ordered TRAIN and held-out TEST data.

    TEST is hashed only for cache integrity; it is not used for D2, admission,
    loss weights, model selection, or fitting.
    """
    digest=hashlib.sha256()
    for tag,x,y in ((b'TRAIN',xtr,ytr),(b'TEST',xt,yt)):
        digest.update(tag)
        digest.update(np.asarray(pd.util.hash_pandas_object(x,index=False).to_numpy(),dtype=np.uint64).tobytes()
                      if isinstance(x,pd.DataFrame) else np.ascontiguousarray(x).tobytes())
        digest.update(np.ascontiguousarray(np.asarray(y,dtype=np.int64)).tobytes())
    return digest.hexdigest()


def apply_one_defect(x, y, scenario, seed):
    """Mirror the benchmark's Adult injection strength; ONE operation per case.

    Missingness uses replacement sampling as in the original Adult script.
    Duplicate selects 15% rows and copies their source records AND labels.
    Invalid targets 6% of rows in hours-per-week; labels flip 20%.
    """
    rng=np.random.default_rng(seed)
    z=x.copy(deep=True)
    labels=np.asarray(y).copy()
    n=len(labels)
    if scenario=='missing':
        count=max(1,int(n*len(z.columns)*0.12))
        rows=rng.integers(0,n,size=count)
        cols=rng.integers(0,len(z.columns),size=count)
        for row,col in zip(rows,cols):
            z.iat[int(row),int(col)] = np.nan
    elif scenario=='duplicate':
        if n>2:
            indices=rng.choice(n,max(1,int(n*0.15)),replace=False)
            sources=rng.choice(n,len(indices),replace=True)
            z.iloc[indices]=z.iloc[sources].to_numpy()
            labels[indices]=labels[sources]
    elif scenario=='invalid':
        ids=rng.choice(n,max(1,int(n*0.06)),replace=False)
        z.iloc[ids,z.columns.get_loc('hours-per-week')]=-999
    elif scenario=='label_flip':
        ids=rng.choice(n,max(1,int(n*0.20)),replace=False)
        labels[ids]=1-labels[ids]
    else:
        raise ValueError(scenario)
    return z, labels


def _save_csv(path, data):
    if not data: raise ValueError('No rows to save')
    with open(path,'w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=list(data[0]));writer.writeheader();writer.writerows(data)


def _archive_cases(archive):
    decision_records={}
    with ZipFile(archive) as z:
        for path in z.namelist():
            if '/decisions_adult_seed' not in path or not path.endswith('_none.json'):
                continue
            obj=json.loads(z.read(path))
            if obj.get('dataset')!='adult': raise ValueError('Unexpected dataset')
            key=(float(obj['training_config']['alpha']),int(obj['seed']),
                 'degraded' if 'degraded' in path else 'clean')
            if key in decision_records: raise ValueError(f'Duplicate case {key}')
            decision_records[key]=obj
    expected={(a,seed,c) for a in (0.5,0.1) for seed in range(42,47) for c in ('clean','degraded')}
    if set(decision_records)!=expected:
        raise ValueError(f'Archive is missing cases {expected-set(decision_records)}')
    return decision_records


def run(archive, adult_data, output):
    out=Path(output);out.mkdir(parents=True,exist_ok=True)
    cases=_archive_cases(archive)
    cfg=SimpleNamespace(test_fraction=.20,global_split_seed=2026,min_client_samples=16,
                        max_partition_repair_fraction=.10)
    loaded=load_dataset('adult',adult_csv=str(adult_data))
    if len(loaded[1])!=32561:
        raise ValueError(f'Adult source has {len(loaded[1])} records, expected 32561. Supply original UCI adult.data.')
    xtr,ytr,ref,xt,yt=split_global('adult',loaded,cfg,42)
    fingerprint=split_fingerprint(xtr,ytr,xt,yt)
    fingerprints={d['global_train_test_sha256'] for d in cases.values()}
    if fingerprints!={fingerprint}:
        raise ValueError('Data/split fingerprint does NOT match archived benchmark. Do not mix results. '
                         f'Observed {fingerprint} vs archived {fingerprints}.')
    print('PASS: data and original TRAIN/TEST split fingerprints match the archive.',flush=True)
    ids=[f'C{i:03d}' for i in range(50)]
    rows=[]; controls=[]
    for alpha in (0.5,0.1):
        for seed in range(42,47):
            clean=cases[(alpha,seed,'clean')]
            degraded=cases[(alpha,seed,'degraded')]
            policy=Policy(**clean['policy'])
            client_ids=set(clean['decisions'])
            if client_ids!=set(ids):raise ValueError('Unexpected client IDs')
            if any(clean['decisions'][cid]['factor_scores'][f]!=degraded['decisions'][cid]['factor_scores'][f]
                   for cid in ids for f in ALL_FACTORS if f not in D2_FACTORS):
                raise ValueError('Documentary scoring changed unexpectedly across clean/degraded')
            idx,diag=dirichlet_partition(ytr,50,alpha,seed+31,min_samples=16,
                                        max_repair_fraction=.10,return_diagnostics=True)
            selected=set(degraded['degraded_ids'])
            expected=set(np.random.default_rng(seed+339).choice(ids,size=15,replace=False).tolist())
            if selected!=expected:raise ValueError('Original targeted client IDs did not reproduce')
            # Control group is all original clean clients; no remeasurement needed.
            for cid in ids:
                rec=clean['decisions'][cid]
                score=rec['factor_scores']
                controls.append(dict(alpha=alpha,seed=seed,client=cid,
                    targeted=int(cid in selected),original_admitted=int(rec['accepted']),
                    original_path=rec['path'],original_hps=rec['hps'],
                    original_D2=rec['dimensions']['D2'],
                    original_distribution_score=score['feature_distribution_consistency'],
                    original_distribution_zero=int(score['feature_distribution_consistency']==0),
                    original_any_D2_zero=int(any(score[f]==0 for f in D2_FACTORS))))
            for cid in sorted(selected):
                client_index=int(cid[1:])
                x=xtr.iloc[idx[client_index]].reset_index(drop=True)
                y=np.asarray(ytr)[idx[client_index]].copy()
                saved=clean['decisions'][cid]
                original_raw=adult_train_metrics(x,y,ref)
                for key in D2_FACTORS:
                    if not np.isclose(original_raw[key],saved['train_d2_raw_metrics'][key],atol=1e-9,rtol=0):
                        raise AssertionError(f'Mismatch in clean D2 raw metric for {alpha}/{seed}/{cid}/{key}: '
                                             f'{original_raw[key]} versus {saved["train_d2_raw_metrics"][key]}')
                baseline_scores=saved['factor_scores']
                if d2_from_metrics(original_raw)!={k:baseline_scores[k] for k in D2_FACTORS}:
                    raise AssertionError(f'Mismatch clean D2 score for {alpha}/{seed}/{cid}')
                policy_check=evaluate(baseline_scores,policy)
                if policy_check.accepted != saved['accepted']:
                    raise AssertionError('Original decision cannot be reproduced')
                for scenario in ('missing','duplicate','invalid','label_flip'):
                    # Deterministic independent random stream per scenario and client.
                    scenario_index=('missing','duplicate','invalid','label_flip').index(scenario)
                    defect_seed=seed*10007+client_index*17+scenario_index*100000
                    changed_x,changed_y=apply_one_defect(x,y,scenario,defect_seed)
                    raw=adult_train_metrics(changed_x,changed_y,ref)
                    scores=d2_from_metrics(raw)
                    full={**baseline_scores, **scores}
                    assessment=evaluate(full,policy)
                    changed_factors=[k for k in D2_FACTORS if scores[k]!=baseline_scores[k]]
                    worsened=[k for k in D2_FACTORS if scores[k]<baseline_scores[k]]
                    rows.append(dict(alpha=alpha,seed=seed,client=cid,scenario=scenario,
                        n_records=len(y),baseline_admitted=int(saved['accepted']),
                        scenario_admitted=int(assessment.accepted),
                        admitted_to_rejected=int(saved['accepted'] and not assessment.accepted),
                        rejected_to_admitted=int(not saved['accepted'] and assessment.accepted),
                        baseline_path=saved['path'],scenario_path=assessment.path,
                        baseline_hps=float(saved['hps']),scenario_hps=assessment.hps,
                        delta_hps=round(assessment.hps-float(saved['hps']),8),
                        baseline_D2=float(saved['dimensions']['D2']),scenario_D2=assessment.dimensions['D2'],
                        changed_D2_count=len(changed_factors),worsened_D2_count=len(worsened),
                        any_D2_worsened=int(bool(worsened)),
                        target_D2_zero=int(any(scores[f]==0 for f in D2_FACTORS)),
                        baseline_D2_zero=int(any(baseline_scores[f]==0 for f in D2_FACTORS)),
                        changed_factor_names=';'.join(changed_factors),
                        worsened_factor_names=';'.join(worsened),
                        baseline_distribution_score=baseline_scores['feature_distribution_consistency'],
                        scenario_distribution_score=scores['feature_distribution_consistency'],
                        baseline_label_integrity_score=baseline_scores['label_integrity'],
                        scenario_label_integrity_score=scores['label_integrity'],
                        raw_missing_before=round(original_raw['completeness'],6),
                        raw_missing_after=round(raw['completeness'],6),
                        raw_duplicate_before=round(original_raw['duplication_rate'],6),
                        raw_duplicate_after=round(raw['duplication_rate'],6),
                        raw_invalid_before=round(original_raw['value_validity_error_rate'],6),
                        raw_invalid_after=round(raw['value_validity_error_rate'],6)))
            print(f'Finished alpha={alpha}, seed={seed}: {len(selected)} original targeted clients.',flush=True)
    _save_csv(out/'B_client_level_defect_results.csv',rows)
    _save_csv(out/'B_clean_noniid_controls.csv',controls)
    summary=[]
    group=defaultdict(list)
    for row in rows:group[(row['alpha'],row['scenario'])].append(row)
    for (alpha,scenario),items in sorted(group.items()):
        total=len(items)
        eligible=sum(row['baseline_admitted'] for row in items)
        summary.append(dict(alpha=alpha,scenario=scenario,cases=total,
            initially_admitted=eligible,initially_rejected=total-eligible,
            any_D2_score_declined=sum(row['any_D2_worsened'] for row in items),
            score_decline_rate=round(sum(row['any_D2_worsened'] for row in items)/total,4),
            admitted_to_rejected=sum(row['admitted_to_rejected'] for row in items),
            initial_admitted_rejection_rate=round(sum(row['admitted_to_rejected'] for row in items)/eligible,4) if eligible else '',
            rejected_to_admitted=sum(row['rejected_to_admitted'] for row in items),
            mean_D2_change=round(float(np.mean([row['scenario_D2']-row['baseline_D2'] for row in items])),6),
            mean_HPS_change=round(float(np.mean([row['delta_hps'] for row in items])),6)))
    _save_csv(out/'B_summary_by_defect.csv',summary)
    controls_summary=[]
    for a in (.5,.1):
        arr=[row for row in controls if row['alpha']==a]
        zero=sum(row['original_any_D2_zero'] for row in arr)
        dist_zero=sum(row['original_distribution_zero'] for row in arr)
        controls_summary.append(dict(alpha=a,clean_client_assessments=len(arr),
            rejected_in_clean=sum(not row['original_admitted'] for row in arr),
            clean_any_D2_zero=zero,
            clean_distribution_score_zero=dist_zero,
            clean_D2_zero_rate=round(zero/len(arr),5),
            clean_distribution_zero_rate=round(dist_zero/len(arr),5)))
    _save_csv(out/'B_clean_noniid_summary.csv',controls_summary)
    report=[
        '# Step 2 — HPS single-defect data-quality assessment (Adult)', '',
        'Dataset and split fingerprints checked against the original Adult archive.',
        'Original documentary scores, admission policy, partitions, and global TRAIN reference retained.',
        'Fifteen originally targeted clients per alpha/seed; 4 separate defect scenarios each.',
        'Counts are correlated repeated virtual-client/seed assessments, NOT independent institutions.',
        'No model retraining, held-out TEST use for scoring, or real provenance verification.',
        '', '## Raw scenario counts', '',
        '| Alpha | Defect | Cases | D2 declines | Accepted→rejected | Rejected→accepted | Mean HPS change |',
        '|---|---|---:|---:|---:|---:|---:|']
    for r in summary:
        report.append(f'| {r["alpha"]} | {r["scenario"]} | {r["cases"]} | '
                      f'{r["any_D2_score_declined"]} | {r["admitted_to_rejected"]} | '
                      f'{r["rejected_to_admitted"]} | {r["mean_HPS_change"]:+.4f} |')
    report += ['', '## False-exclusion warning',
        'The clean cases are NOT independently labeled as legally eligible clients.',
        'Zero distribution-consistency scores among clean clients show possible non-IID exclusions,',
        'but are not validated false-positive rates without an independent suitability label.',
        '', '## Interpretation',
        '- Valid-class label flipping is deliberately not measurable by the current D2 label-integrity check.',
        '- Changes to the distribution-consistency factor can reflect legitimate heterogeneity as well as injected defects.',
        '- Combined original degradation is not analyzed here; each scenario modifies only one defect category.',
        '- Do not claim that these tests authenticate documentary evidence or legal compliance.',
        '- Do not choose HPS weights/thresholds by maximizing held-out TEST outcomes.']
    (out/'B_README.md').write_text('\n'.join(report)+'\n',encoding='utf-8')
    print('\n'.join(report[:19]))
    return out



# ============================================================================
# SIMPLE COLAB EXECUTION: upload this file, adult.data, and the Adult results ZIP.
# In a Colab cell: %run 02_Run_HPS_Individual_Defect_Tests.py
# ============================================================================

def _run_one_click():
    from pathlib import Path
    import zipfile

    root = Path('/content') if Path('/content').is_dir() else Path.cwd()
    adult_archive = root/'TADP_HPS28_Adult_Phased_Results_v5.zip'
    if not adult_archive.exists():
        matches=sorted(root.glob('TADP_HPS28_Adult_Phased_Results_v5*.zip'))
        if len(matches)!=1:
            raise FileNotFoundError('Upload TADP_HPS28_Adult_Phased_Results_v5.zip in the Colab Files panel.')
        adult_archive=matches[0]
    adult_data = root/'adult.data'
    if not adult_data.exists():
        candidates=[p for p in (root/'adult.csv',root/'adult_data.csv') if p.exists()]
        if len(candidates)!=1:
            raise FileNotFoundError('Upload the original UCI adult.data (32,561 records) in the Colab Files panel.')
        adult_data=candidates[0]
    result_dir = root/'STEP2_HPS_Defect_Results'
    print('EXPERIMENT 2: FOUR ISOLATED DATA-QUALITY DEFECTS (no model training)')
    print('Using:', adult_archive.name, 'and', adult_data.name, flush=True)
    run(adult_archive, adult_data, result_dir)
    bundle=root/'STEP2_HPS_Defect_Results.zip'
    with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for entry in sorted(result_dir.rglob('*')):
            if entry.is_file():
                z.write(entry, entry.relative_to(result_dir.parent))
    print('SUCCESS. Download:', str(bundle), flush=True)
    print('In Colab, click the Files (folder) icon and download the ZIP.')
    return bundle

if __name__ == '__main__':
    _run_one_click()


EXPERIMENT 2: FOUR ISOLATED DATA-QUALITY DEFECTS (no model training)
Using: TADP_HPS28_Adult_Phased_Results_v5.zip and adult.data
PASS: data and original TRAIN/TEST split fingerprints match the archive.
Finished alpha=0.5, seed=42: 15 original targeted clients.
Finished alpha=0.5, seed=43: 15 original targeted clients.
Finished alpha=0.5, seed=44: 15 original targeted clients.
Finished alpha=0.5, seed=45: 15 original targeted clients.
Finished alpha=0.5, seed=46: 15 original targeted clients.
Finished alpha=0.1, seed=42: 15 original targeted clients.
Finished alpha=0.1, seed=43: 15 original targeted clients.
Finished alpha=0.1, seed=44: 15 original targeted clients.
Finished alpha=0.1, seed=45: 15 original targeted clients.
Finished alpha=0.1, seed=46: 15 original targeted clients.
# Step 2 — HPS single-defect data-quality assessment (Adult)

Dataset and split fingerprints checked against the original Adult archive.
Original documentary scores, admission policy, partitions, and global 